# dsfb-tmtr Colab Reproduction

This notebook builds the `dsfb-tmtr` crate from source, runs deterministic TMTR simulations, loads the newest output directory, and generates a paper-aligned figure suite.

The results shown here are empirical outputs from a deterministic reference implementation. They are intended for reproducibility and inspection, not deployment.

## Notebook Flow

1. Environment setup
2. Locate the repository and crate
3. Build the crate from source
4. Run deterministic simulations
5. Locate the newest output directory
6. Load exported artifacts
7. Generate publication-quality figures
8. Print numerical summary metrics
9. Interpret the empirical findings
10. List generated figure files

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True, cwd=cwd)


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "matplotlib"], check=True)

if shutil.which("rustc") is None:
    run_command(["bash", "-lc", "curl https://sh.rustup.rs -sSf | sh -s -- -y"])

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
run_command(["rustc", "--version"])
run_command(["cargo", "--version"])


## Locate the Repository

The notebook searches for a repository root containing `crates/dsfb-tmtr/Cargo.toml`. If it cannot find one locally, it automatically clones `https://github.com/infinityabundance/dsfb.git` into `/content/dsfb` and can optionally honor `DSFB_REPO_URL` and `DSFB_REPO_REF`.

In [ ]:
REPO_ROOT_OVERRIDE = os.environ.get("DSFB_REPO_ROOT", "").strip()
DEFAULT_REPO_URL = "https://github.com/infinityabundance/dsfb.git"
REPO_URL = os.environ.get("DSFB_REPO_URL", DEFAULT_REPO_URL).strip()
REPO_REF_OVERRIDE = os.environ.get("DSFB_REPO_REF", "").strip()
CLONE_DIR = Path("/content/dsfb")


def has_repo_root(path: Path) -> bool:
    return (path / "crates" / "dsfb-tmtr" / "Cargo.toml").exists()


def find_repo_root():
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content"),
        Path("/content/dsfb"),
        Path("/content/repo"),
    ])
    if Path("/content").exists():
        for child in sorted(Path("/content").iterdir()):
            candidates.append(child)
            if child.is_dir():
                candidates.extend(sorted(child.iterdir()))
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except FileNotFoundError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_repo_root(candidate):
            return candidate
    return None


def candidate_refs():
    refs = []
    for ref in [REPO_REF_OVERRIDE, "codex-dsfb-tmtr-prototype", "main"]:
        if ref and ref not in refs:
            refs.append(ref)
    return refs


def try_checkout_ref(repo_dir: Path, ref: str) -> bool:
    print(f"Trying repo ref: {ref}")
    fetch = subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", ref],
        cwd=repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if fetch.returncode != 0:
        output = fetch.stdout.strip()
        if output:
            print(output)
        return False
    checkout = subprocess.run(
        ["git", "checkout", "FETCH_HEAD"],
        cwd=repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if checkout.returncode != 0:
        output = checkout.stdout.strip()
        if output:
            print(output)
        return False
    print(f"Checked out ref: {ref}")
    return True


def ensure_repo_clone() -> Path:
    if has_repo_root(CLONE_DIR) and not REPO_REF_OVERRIDE:
        return CLONE_DIR
    if CLONE_DIR.exists() and not (CLONE_DIR / ".git").exists():
        shutil.rmtree(CLONE_DIR)
    if not CLONE_DIR.exists():
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)])
    for ref in candidate_refs():
        if try_checkout_ref(CLONE_DIR, ref) and has_repo_root(CLONE_DIR):
            return CLONE_DIR
    if has_repo_root(CLONE_DIR):
        return CLONE_DIR
    raise FileNotFoundError(
        "The cloned repository does not contain crates/dsfb-tmtr/Cargo.toml. "
        "Set DSFB_REPO_URL or DSFB_REPO_REF to the correct repository state and rerun this cell."
    )


repo_root = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root()
if repo_root is None:
    repo_root = ensure_repo_clone()

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate a repository root containing crates/dsfb-tmtr/Cargo.toml. "
        "Set DSFB_REPO_ROOT, DSFB_REPO_URL, or DSFB_REPO_REF before rerunning this cell."
    )

crate_dir = repo_root / "crates" / "dsfb-tmtr"
output_root = repo_root / "output-dsfb-tmtr"

print("Repo root:", repo_root)
print("Crate dir:", crate_dir)
print("Output root:", output_root)


## Build and Run the Deterministic Simulation

The crate is rebuilt from source on every notebook execution. The simulation run creates a new timestamped directory under `output-dsfb-tmtr/` and never overwrites prior runs.

In [ ]:
SCENARIO = "all"
N_STEPS = 600

run_command(["cargo", "build", "--manifest-path", str(crate_dir / "Cargo.toml"), "--release"])
run_command([
    "cargo",
    "run",
    "--manifest-path",
    str(crate_dir / "Cargo.toml"),
    "--release",
    "--",
    "--scenario",
    SCENARIO,
    "--n-steps",
    str(N_STEPS),
    "--output-root",
    str(output_root),
])


## Locate the Newest Output Directory and Load Artifacts

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

run_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
if not run_dirs:
    raise FileNotFoundError(f"No run directories found under {output_root}")

run_dir = run_dirs[-1]
figures_dir = run_dir / "figures"
figures_dir.mkdir(exist_ok=True)

required_files = [
    "run_manifest.json",
    "config.json",
    "scenario_summary.csv",
    "trajectories.csv",
    "trust_timeseries.csv",
    "residuals.csv",
    "correction_events.csv",
    "prediction_tubes.csv",
    "causal_edges.csv",
    "causal_metrics.csv",
    "notebook_ready_summary.json",
]

loaded = {}
for name in required_files:
    path = run_dir / name
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact: {path}")
    if name.endswith(".csv"):
        loaded[name] = pd.read_csv(path)
    else:
        with path.open() as handle:
            loaded[name] = json.load(handle)

print("Loaded files:")
for name in required_files:
    print(" -", name)

scenario_summary = loaded["scenario_summary.csv"]
trajectories = loaded["trajectories.csv"]
trust_timeseries = loaded["trust_timeseries.csv"]
residuals = loaded["residuals.csv"]
correction_events = loaded["correction_events.csv"]
prediction_tubes = loaded["prediction_tubes.csv"]
causal_edges = loaded["causal_edges.csv"]
causal_metrics = loaded["causal_metrics.csv"]
notebook_summary = loaded["notebook_ready_summary.json"]

primary_scenario = notebook_summary["primary_scenario"]
prediction_scenario = "forward_prediction" if "forward_prediction" in set(scenario_summary["scenario"]) else primary_scenario

print("Active run directory:", run_dir)
print("Figure output directory:", figures_dir)
print("Primary scenario:", primary_scenario)
print("Prediction scenario:", prediction_scenario)


## Generate Publication-Quality Figures

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10.0, 4.8),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "savefig.dpi": 250,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linewidth": 0.6,
})

figures_manifest = []
summary_by_scenario = scenario_summary.set_index("scenario")


def mode_level(frame, scenario_name, mode, level):
    data = frame[
        (frame["scenario"] == scenario_name)
        & (frame["mode"] == mode)
        & (frame["observer_level"] == level)
    ].copy()
    if "time_index" in data.columns:
        data = data.sort_values("time_index")
    return data


def finalize_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    ordered = dict(zip(labels, handles))
    ax.legend(ordered.values(), ordered.keys(), frameon=False)


def save_figure(fig, filename, purpose, sources):
    path = figures_dir / filename
    fig.tight_layout()
    fig.savefig(path, dpi=250, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    figures_manifest.append({
        "file": filename,
        "purpose": purpose,
        "source_data_files": sources,
    })


primary_summary = summary_by_scenario.loc[primary_scenario]
trajectory_baseline = mode_level(trajectories, primary_scenario, "baseline", 1)
trajectory_tmtr = mode_level(trajectories, primary_scenario, "tmtr", 1)
residual_baseline = mode_level(residuals, primary_scenario, "baseline", 1)
residual_tmtr = mode_level(residuals, primary_scenario, "tmtr", 1)
trust_primary = trust_timeseries[
    (trust_timeseries["scenario"] == primary_scenario)
    & (trust_timeseries["mode"] == "tmtr")
].sort_values(["observer_level", "time_index"])
corrections_primary = correction_events[correction_events["scenario"] == primary_scenario].copy()

# Figure 1
fig, ax = plt.subplots()
ax.plot(trajectory_baseline["time_index"], trajectory_baseline["ground_truth"], label="Ground truth", color="black", linewidth=2.2)
ax.plot(trajectory_baseline["time_index"], trajectory_baseline["estimate"], label="Baseline", linewidth=1.7)
ax.plot(trajectory_tmtr["time_index"], trajectory_tmtr["estimate"], label="TMTR", linewidth=1.7)
ax.axvspan(primary_summary["degraded_start"], primary_summary["degraded_end"], color="0.90", label="Degraded interval")
ax.axvspan(primary_summary["degraded_end"], primary_summary["refinement_end"], color="0.82", alpha=0.35, label="Refinement interval")
ax.set_title("Figure 1. Trajectory Reconstruction Comparison")
ax.set_xlabel("Time")
ax.set_ylabel("State value")
finalize_legend(ax)
save_figure(fig, "01_trajectory_reconstruction.png", "True versus baseline versus TMTR trajectory reconstruction over the degraded interval.", ["trajectories.csv", "scenario_summary.csv"])

# Figure 2
fig, ax = plt.subplots()
ax.plot(residual_baseline["time_index"], residual_baseline["abs_residual"], label="Baseline error", linewidth=1.7)
ax.plot(residual_tmtr["time_index"], residual_tmtr["abs_residual"], label="TMTR error", linewidth=1.7)
ax.axvspan(primary_summary["degraded_start"], primary_summary["refinement_end"], color="0.90", alpha=0.6, label="Correction-active interval")
ax.set_title("Figure 2. Retroactive Error Reduction")
ax.set_xlabel("Time")
ax.set_ylabel("Absolute error")
finalize_legend(ax)
save_figure(fig, "02_error_reduction.png", "Absolute reconstruction error comparison during retroactive refinement.", ["residuals.csv", "scenario_summary.csv"])

# Figure 3
fig, ax = plt.subplots()
for level in sorted(trust_primary["observer_level"].unique()):
    level_frame = trust_primary[trust_primary["observer_level"] == level]
    ax.plot(level_frame["time_index"], level_frame["trust"], label=f"Observer level {level}", linewidth=1.7)
if not corrections_primary.empty:
    anchors = np.sort(corrections_primary["anchor_time"].unique())
    marker_step = max(1, len(anchors) // 12)
    for anchor in anchors[::marker_step]:
        ax.axvline(anchor, color="0.90", linewidth=0.6)
ax.set_ylim(0.0, 1.02)
ax.set_title("Figure 3. Trust Envelopes by Observer Level")
ax.set_xlabel("Time")
ax.set_ylabel("Trust value")
finalize_legend(ax)
save_figure(fig, "03_trust_envelopes.png", "Trust evolution across the three-level observer hierarchy with correction anchors marked.", ["trust_timeseries.csv", "correction_events.csv"])

# Figure 4
fig, ax = plt.subplots()
ax.plot(residual_baseline["time_index"], residual_baseline["abs_residual"], label="Baseline residual", linewidth=1.7)
ax.plot(residual_tmtr["time_index"], residual_tmtr["abs_residual"], label="TMTR residual", linewidth=1.7)
ax.axhline(0.06, color="0.35", linestyle="--", linewidth=1.0, label="Recovery threshold")
ax.set_title("Figure 4. Residual Convergence")
ax.set_xlabel("Time")
ax.set_ylabel("Residual magnitude")
finalize_legend(ax)
save_figure(fig, "04_residual_convergence.png", "Residual stabilization under baseline and TMTR trajectories.", ["residuals.csv"])

# Figure 5
tube_frame = prediction_tubes[prediction_tubes["scenario"] == prediction_scenario].copy()
width_by_anchor = tube_frame.groupby(["mode", "anchor_time"])["width"].mean().unstack("mode").dropna()
if width_by_anchor.empty:
    raise RuntimeError("Prediction tube data is unavailable for the selected scenario.")
width_by_anchor["improvement"] = width_by_anchor["baseline"] - width_by_anchor["tmtr"]
selected_anchor = int(width_by_anchor["improvement"].idxmax())
tube_baseline = tube_frame[(tube_frame["mode"] == "baseline") & (tube_frame["anchor_time"] == selected_anchor)].sort_values("future_time")
tube_tmtr = tube_frame[(tube_frame["mode"] == "tmtr") & (tube_frame["anchor_time"] == selected_anchor)].sort_values("future_time")
fig, ax = plt.subplots()
ax.fill_between(tube_baseline["future_time"], tube_baseline["lower"], tube_baseline["upper"], alpha=0.25, label="Baseline tube")
ax.fill_between(tube_tmtr["future_time"], tube_tmtr["lower"], tube_tmtr["upper"], alpha=0.35, label="TMTR tube")
ax.plot(tube_baseline["future_time"], tube_baseline["ground_truth"], color="black", linewidth=2.0, label="Ground truth")
ax.plot(tube_baseline["future_time"], tube_baseline["center"], linewidth=1.2, linestyle="--", label="Baseline center")
ax.plot(tube_tmtr["future_time"], tube_tmtr["center"], linewidth=1.2, linestyle="--", label="TMTR center")
ax.set_title(f"Figure 5. Forward Prediction Tube Comparison (anchor={selected_anchor})")
ax.set_xlabel("Future time")
ax.set_ylabel("Predicted state")
finalize_legend(ax)
save_figure(fig, "05_prediction_tubes.png", "Baseline versus TMTR deterministic prediction tubes at the strongest tightening anchor.", ["prediction_tubes.csv"])

# Figure 6
depth_counts = corrections_primary["recursion_depth"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8.2, 4.6))
ax.bar(depth_counts.index.astype(str), depth_counts.values, color="0.45")
ax.set_title("Figure 6. Recursion Depth Distribution")
ax.set_xlabel("Recursion depth")
ax.set_ylabel("Correction event count")
ax.text(0.98, 0.96, f"Max depth: {int(corrections_primary['recursion_depth'].max())}", transform=ax.transAxes, ha="right", va="top", bbox={"facecolor": "white", "edgecolor": "0.7"})
save_figure(fig, "06_recursion_depth.png", "Observed bounded recursion depth across TMTR correction events.", ["correction_events.csv"])

# Figure 7
fig, ax = plt.subplots()
for target_level, color in [(1, "0.25"), (2, "0.60")]:
    magnitudes = corrections_primary[corrections_primary["target_level"] == target_level]["correction_magnitude"].abs()
    ax.hist(magnitudes, bins=32, alpha=0.45, label=f"Target level {target_level}", color=color)
ax.set_title("Figure 7. Correction Magnitude Distribution")
ax.set_xlabel("Absolute correction magnitude")
ax.set_ylabel("Count")
finalize_legend(ax)
save_figure(fig, "07_correction_magnitudes.png", "Distribution of TMTR correction magnitudes grouped by target observer level.", ["correction_events.csv"])

# Figure 8
causal_primary = causal_metrics[causal_metrics["scenario"] == primary_scenario].copy()
metric_order = ["edge_count", "reachable_nodes_from_anchor", "max_path_length"]
metric_labels = ["Edges", "Reachability", "Max path length"]
causal_plot = causal_primary[causal_primary["metric"].isin(metric_order)].pivot(index="metric", columns="mode", values="value").loc[metric_order]
x = np.arange(len(metric_order))
width = 0.36
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.bar(x - width / 2, causal_plot["baseline"], width, label="Baseline")
ax.bar(x + width / 2, causal_plot["tmtr"], width, label="TMTR")
ax.set_yscale("log")
ax.set_xticks(x, metric_labels)
ax.set_title("Figure 8. Causal Consistency Summary")
ax.set_xlabel("Causal metric")
ax.set_ylabel("Value (log scale)")
extra_text = (
    f"Cycle count: {int(causal_primary[(causal_primary['mode'] == 'tmtr') & (causal_primary['metric'] == 'cycle_count')]['value'].iloc[0])}\n"
    f"Backward edges: {int(causal_primary[(causal_primary['mode'] == 'tmtr') & (causal_primary['metric'] == 'backward_edge_count')]['value'].iloc[0])}\n"
    f"Local density: {causal_primary[(causal_primary['mode'] == 'tmtr') & (causal_primary['metric'] == 'local_window_edge_density')]['value'].iloc[0]:.4f}"
)
ax.text(0.98, 0.98, extra_text, transform=ax.transAxes, ha="right", va="top", bbox={"facecolor": "white", "edgecolor": "0.7"})
finalize_legend(ax)
save_figure(fig, "08_causal_consistency.png", "Grouped causal summary metrics with explicit cycle and backward-edge checks.", ["causal_metrics.csv"])

# Figure 9
comparison_categories = ["RMSE", "Recovery time", "Avg tube width", "Retro interval MAE"]
baseline_values = [
    primary_summary["baseline_rmse"],
    primary_summary["baseline_recovery_time"],
    primary_summary["baseline_avg_tube_width"],
    primary_summary["retro_baseline_mae"],
]
tmtr_values = [
    primary_summary["tmtr_rmse"],
    primary_summary["tmtr_recovery_time"],
    primary_summary["tmtr_avg_tube_width"],
    primary_summary["retro_tmtr_mae"],
]
x = np.arange(len(comparison_categories))
fig, ax = plt.subplots(figsize=(9.6, 4.8))
ax.bar(x - width / 2, baseline_values, width, label="Baseline")
ax.bar(x + width / 2, tmtr_values, width, label="TMTR")
ax.set_xticks(x, comparison_categories)
ax.set_title("Figure 9. Baseline vs TMTR Summary Panel")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
summary_text = (
    f"Monotonicity violations: {int(primary_summary['monotonicity_violations'])}\n"
    f"Cycle count: {int(primary_summary['tmtr_cycle_count'])}\n"
    f"Max recursion depth: {int(primary_summary['max_recursion_depth'])}"
)
ax.text(0.98, 0.98, summary_text, transform=ax.transAxes, ha="right", va="top", bbox={"facecolor": "white", "edgecolor": "0.7"})
finalize_legend(ax)
save_figure(fig, "09_summary_comparison.png", "Compact baseline versus TMTR comparison across reconstruction, recovery, and tube metrics.", ["scenario_summary.csv"])

with (figures_dir / "figures_manifest.json").open("w") as handle:
    json.dump(figures_manifest, handle, indent=2)

print(f"Saved {len(figures_manifest)} figures to {figures_dir}")


## Figure 10. Retroactive Correction DAG Visualization

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

retro_scenario = "disturbance_recovery" if "disturbance_recovery" in summary_by_scenario.index else primary_scenario
retro_summary = summary_by_scenario.loc[retro_scenario]

retro_corrections = correction_events[correction_events["scenario"] == retro_scenario].copy()
for column in ["source_level", "target_level", "anchor_time", "corrected_time", "delta_window", "recursion_depth", "iteration"]:
    retro_corrections[column] = retro_corrections[column].astype(int)
for column in ["trust_weight", "correction_magnitude"]:
    retro_corrections[column] = retro_corrections[column].astype(float)
retro_corrections["abs_correction"] = retro_corrections["correction_magnitude"].abs()

retro_candidates = retro_corrections[
    (retro_corrections["target_level"] == 1)
    & (retro_corrections["anchor_time"] > int(retro_summary["degraded_end"]))
    & (retro_corrections["corrected_time"] >= int(retro_summary["degraded_start"]))
    & (retro_corrections["corrected_time"] <= int(retro_summary["degraded_end"]))
].copy()

if retro_candidates.empty:
    retro_candidates = retro_corrections[
        (retro_corrections["target_level"] == 1)
        & (retro_corrections["anchor_time"] > int(retro_summary["degraded_end"]))
    ].copy()

if retro_candidates.empty:
    retro_candidates = retro_corrections.copy()

retro_scores = retro_candidates.groupby(["anchor_time", "source_level", "target_level", "delta_window"], as_index=False).agg(
    refined_start=("corrected_time", "min"),
    refined_end=("corrected_time", "max"),
    total_abs_correction=("abs_correction", "sum"),
    mean_abs_correction=("abs_correction", "mean"),
    mean_trust=("trust_weight", "mean"),
    event_count=("corrected_time", "size"),
)
retro_scores["retro_span"] = retro_scores["anchor_time"] - retro_scores["refined_start"]
retro_scores["score"] = retro_scores["total_abs_correction"] * (
    1.0 + retro_scores["retro_span"] / retro_scores["delta_window"].clip(lower=1)
)
retro_pick = retro_scores.sort_values(
    ["score", "retro_span", "mean_trust", "event_count"],
    ascending=[False, False, False, False],
).iloc[0]

trigger_time = int(retro_pick["anchor_time"])
trigger_level = int(retro_pick["source_level"])
target_level = int(retro_pick["target_level"])
delta_window = int(retro_pick["delta_window"])
refined_start = int(retro_pick["refined_start"])
refined_end = int(retro_pick["refined_end"])
degraded_start = int(retro_summary["degraded_start"])
degraded_end = int(retro_summary["degraded_end"])

window_start = max(0, min(refined_start, degraded_start) - 4)
window_end = min(int(retro_summary["n_steps"]) - 1, max(trigger_time + 6, degraded_end + 4))
display_times = np.arange(window_start, window_end + 1)

retro_traj = trajectories[
    (trajectories["scenario"] == retro_scenario)
    & (trajectories["observer_level"] == target_level)
    & (trajectories["mode"].isin(["baseline", "tmtr"]))
].copy()
retro_traj["time_index"] = retro_traj["time_index"].astype(int)
retro_traj["estimate"] = retro_traj["estimate"].astype(float)
retro_traj_window = retro_traj[
    (retro_traj["time_index"] >= window_start)
    & (retro_traj["time_index"] <= window_end)
].copy()
retro_change = retro_traj_window.pivot(index="time_index", columns="mode", values="estimate")
retro_change = (retro_change.get("tmtr", pd.Series(dtype=float)) - retro_change.get("baseline", pd.Series(dtype=float))).abs()
retro_change = retro_change.reindex(display_times, fill_value=0.0)
change_max = float(retro_change.max()) if len(retro_change) else 0.0
normalized_change = retro_change / change_max if change_max > 0 else retro_change * 0.0

causal_window = causal_edges[
    (causal_edges["scenario"] == retro_scenario)
    & (causal_edges["mode"] == "tmtr")
].copy()
for column in ["source_time", "target_time"]:
    causal_window[column] = causal_window[column].astype(float)
for column in ["source_level", "target_level"]:
    causal_window[column] = causal_window[column].astype(int)

trust_edges = causal_window[
    (causal_window["edge_type"] == "trust_gate")
    & (causal_window["source_time"] >= max(window_start, trigger_time - 4))
    & (causal_window["source_time"] <= trigger_time + 2)
    & (causal_window["target_time"] >= window_start)
    & (causal_window["target_time"] <= window_end)
].copy()
if trust_edges.empty:
    trust_edges = causal_window[
        (causal_window["edge_type"] == "trust_gate")
        & (causal_window["source_time"] >= window_start)
        & (causal_window["source_time"] <= window_end)
        & (causal_window["target_time"] >= window_start)
        & (causal_window["target_time"] <= window_end)
    ].copy()
trust_edges["distance_to_trigger"] = (trust_edges["source_time"] - trigger_time).abs()
trust_edges = trust_edges.sort_values(["distance_to_trigger", "source_time"]).head(6)

level_y = {1: 1.0, 2: 2.0, 3: 3.0}
level_labels = {
    1: "Level 1 trajectory",
    2: "Level 2 aggregate",
    3: "Level 3 high-trust",
}

fig, ax = plt.subplots(figsize=(12.0, 5.0))

if degraded_start <= window_end and degraded_end >= window_start:
    degraded_left = max(window_start, degraded_start) - 0.5
    degraded_right = min(window_end, degraded_end) + 0.5
    ax.axvspan(degraded_left, degraded_right, color="0.96", zorder=0)
    ax.text(
        0.5 * (degraded_left + degraded_right),
        0.60,
        "degraded sensing interval",
        ha="center",
        va="bottom",
        fontsize=9,
        color="0.35",
    )

refined_rect = Rectangle(
    (refined_start - 0.5, level_y[target_level] - 0.34),
    refined_end - refined_start + 1.0,
    0.68,
    facecolor="tab:blue",
    edgecolor="tab:blue",
    alpha=0.14,
    linewidth=1.2,
    zorder=0.5,
)
ax.add_patch(refined_rect)

horizon_start = max(window_start, trigger_time - delta_window)
ax.plot([horizon_start, trigger_time], [3.36, 3.36], color="0.25", linewidth=1.1)
ax.vlines([horizon_start, trigger_time], 3.31, 3.41, color="0.25", linewidth=1.1)
ax.text(
    0.5 * (horizon_start + trigger_time),
    3.43,
    f"bounded recursion horizon δ={delta_window}",
    ha="center",
    va="bottom",
    fontsize=10,
    color="0.25",
)

for level, y in level_y.items():
    ax.plot(display_times, np.full(display_times.size, y, dtype=float), color="0.75", linewidth=1.2, zorder=1)
    if display_times.size >= 2:
        ax.annotate(
            "",
            xy=(display_times[-1], y),
            xytext=(display_times[-2], y),
            arrowprops=dict(arrowstyle="->", color="0.45", lw=1.2),
        )
    ax.scatter(
        display_times,
        np.full(display_times.size, y, dtype=float),
        s=16,
        facecolor="white",
        edgecolor="0.55",
        linewidth=0.7,
        zorder=2,
    )

refined_times = display_times[(display_times >= refined_start) & (display_times <= refined_end)]
if refined_times.size:
    refined_sizes = 45 + 95 * normalized_change.reindex(refined_times, fill_value=0.0).to_numpy()
    ax.scatter(
        refined_times,
        np.full(refined_times.size, level_y[target_level], dtype=float),
        s=refined_sizes,
        facecolor="tab:blue",
        edgecolor="white",
        linewidth=0.6,
        alpha=0.9,
        zorder=3,
    )

for row in trust_edges.itertuples(index=False):
    ax.annotate(
        "",
        xy=(row.target_time, level_y[row.target_level] + 0.05),
        xytext=(row.source_time, level_y[row.source_level] - 0.05),
        arrowprops=dict(
            arrowstyle="-|>",
            color="0.35",
            lw=1.0,
            alpha=0.60,
            shrinkA=6,
            shrinkB=6,
        ),
        zorder=2.5,
    )

ax.axvline(trigger_time, color="tab:orange", alpha=0.18, linewidth=1.0, zorder=1)
ax.scatter(
    [trigger_time],
    [level_y[trigger_level]],
    s=170,
    facecolor="tab:orange",
    edgecolor="black",
    linewidth=1.0,
    zorder=5,
)

ax.annotate(
    f"later higher-trust trigger\n(level {trigger_level}, t={trigger_time})",
    xy=(trigger_time, level_y[trigger_level]),
    xytext=(trigger_time + 2.0, level_y[trigger_level] + 0.55),
    arrowprops=dict(arrowstyle="->", color="tab:orange", lw=1.2),
    bbox=dict(facecolor="white", edgecolor="tab:orange", alpha=0.95),
    ha="left",
    va="center",
)

refined_center = 0.5 * (refined_start + refined_end)
ax.annotate(
    "retroactive refinement annotation\n(not a DAG edge)",
    xy=(refined_center, level_y[target_level] + 0.12),
    xytext=(trigger_time + 2.2, level_y[target_level] - 0.52),
    arrowprops=dict(
        arrowstyle="->",
        linestyle="--",
        color="tab:orange",
        lw=1.3,
        connectionstyle="arc3,rad=0.18",
    ),
    bbox=dict(facecolor="white", edgecolor="tab:orange", alpha=0.95),
    ha="left",
    va="center",
)

ax.text(
    refined_center,
    level_y[target_level] - 0.44,
    f"retroactively refined segment\n[{refined_start}, {refined_end}]",
    ha="center",
    va="top",
    fontsize=9,
    color="tab:blue",
)
ax.text(
    window_start + 0.2,
    3.62,
    "solid lines and arrows = forward DAG edges only",
    ha="left",
    va="center",
    fontsize=10,
    color="0.25",
)

legend_handles = [
    Line2D([0], [0], color="0.60", lw=1.4, label="Forward DAG edge"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:orange", markeredgecolor="black", markersize=8, label="Higher-trust correction trigger"),
    Patch(facecolor="tab:blue", edgecolor="tab:blue", alpha=0.14, label="Retroactively refined interval"),
    Line2D([0], [0], color="tab:orange", lw=1.3, linestyle="--", label="Refinement annotation (not a DAG edge)"),
]
ax.legend(handles=legend_handles, loc="upper left", frameon=False, ncol=2)

tick_step = max(2, int(np.ceil((window_end - window_start) / 10)))
ax.set_xticks(np.arange(window_start, window_end + 1, tick_step))
ax.set_xlim(window_start - 1, window_end + 10)
ax.set_ylim(0.5, 3.8)
ax.set_yticks([1.0, 2.0, 3.0])
ax.set_yticklabels([level_labels[1], level_labels[2], level_labels[3]])
ax.set_title("Figure 10. Retroactive Correction DAG Visualization")
ax.set_xlabel("Event time index")
ax.set_ylabel("Observer layer")
ax.grid(axis="y", alpha=0.12)
ax.grid(axis="x", alpha=0.08)

save_figure(
    fig,
    "10_retroactive_correction_dag.png",
    "Temporal DAG visualization showing retroactive trajectory refinement triggered by later higher-trust observations without introducing backward causal edges.",
    ["causal_edges.csv", "correction_events.csv", "trajectories.csv", "scenario_summary.csv"],
)

with (figures_dir / "figures_manifest.json").open("w") as handle:
    json.dump(figures_manifest, handle, indent=2)

print(f"Updated figures manifest: {figures_dir / 'figures_manifest.json'}")


This visualization illustrates that TMTR retroactively refines earlier trajectory segments using later higher-trust observations, while the underlying causal graph itself remains forward-directed and acyclic. The dashed annotation marks refinement influence only; the solid DAG edges remain forward in time.

## Numerical Summary and Interpretation

In [ ]:
primary_summary = summary_by_scenario.loc[primary_scenario]
percent_improvement = 0.0 if primary_summary["baseline_rmse"] == 0 else (primary_summary["baseline_rmse"] - primary_summary["tmtr_rmse"]) / primary_summary["baseline_rmse"] * 100.0

print(f"baseline reconstruction error: {primary_summary['baseline_rmse']:.6f}")
print(f"TMTR reconstruction error: {primary_summary['tmtr_rmse']:.6f}")
print(f"percent improvement: {percent_improvement:.2f}%")
print(f"baseline average prediction tube width: {primary_summary['baseline_avg_tube_width']:.6f}")
print(f"TMTR average prediction tube width: {primary_summary['tmtr_avg_tube_width']:.6f}")
print(f"max recursion depth: {int(primary_summary['max_recursion_depth'])}")
print(f"monotonicity violations count: {int(primary_summary['monotonicity_violations'])}")
print(f"cycle count: {int(primary_summary['tmtr_cycle_count'])}")
print(f"active output directory: {run_dir}")
print(f"figure output directory: {figures_dir}")

display(Markdown(
    f"""
## Interpretation

Within this deterministic reference implementation, the simulations are consistent with trust-monotone temporal recursion improving reconstruction accuracy over degraded intervals while preserving bounded recursion and causal consistency.

The empirical results support bounded retroactive refinement, tighter forward prediction tubes, zero detected trust-monotonicity violations, and zero detected cycle creation in the exported causal summaries for the active run at `{run_dir}`.
    """
))

print("Generated figure files:")
for item in figures_manifest:
    print(f" - {item['file']}: {item['purpose']}")
print(" - figures_manifest.json: figure metadata and source file provenance")
